# Bab 6: Statistical Machine Learning

Statistical Machine Learning menggabungkan kekuatan statistik tradisional dengan algoritma modern untuk melakukan prediksi yang sangat akurat.

Dalam bab ini, kita akan mempelajari:
1. K-Nearest Neighbors (KNN).
2. Tree Models (Decision Trees).
3. Bagging dan Random Forest.
4. Boosting (XGBoost, LightGBM).
5. Hyperparameter Tuning.

## 1. K-Nearest Neighbors (KNN)

KNN adalah algoritma berbasis instansi yang melakukan prediksi berdasarkan kemiripan (jarak) dengan tetangga terdekatnya.

### Konsep Utama:
- **K**: Jumlah tetangga yang dipertimbangkan. K yang kecil cenderung overfitting, K yang besar cenderung underfitting.
- **Metrik Jarak**: Euclidean, Manhattan, atau Minkowski.
- **Standardisasi**: Sangat penting karena KNN sangat peka terhadap skala fitur.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X, y = data.data, data.target

scaler = StandardScaler()
X_std = scaler.fit_transform(X)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_std, y)
print(f"Akurasi KNN: {knn.score(X_std, y):.2f}")

## 2. Tree Models (Decision Trees)

Pohon keputusan membagi ruang fitur menjadi serangkaian persegi panjang untuk membuat prediksi.

### Algoritma Pemisahan:
- **Gini Impurity**: Mengukur seberapa sering elemen yang dipilih secara acak akan salah diklasifikasikan.
- **Entropy (Information Gain)**: Mengukur ketidakpastian dalam dataset.

### Kelebihan:
- Sangat mudah diinterpretasikan (White Box model).
- Tidak memerlukan scaling data.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

tree = DecisionTreeClassifier(max_depth=3)
tree.fit(X, y)

plt.figure(figsize=(20, 10))
plot_tree(tree, feature_names=data.feature_names, filled=True)
plt.show()

## 3. Ensemble Learning: Bagging dan Random Forest

Ensemble learning menggabungkan banyak model untuk menghasilkan prediksi yang lebih kuat.

### A. Bagging (Bootstrap Aggregating)
Melatih banyak model pada sampel bootstrap yang berbeda dan merata-ratakan hasilnya.

### B. Random Forest
Pengembangan dari bagging yang juga mengacak fitur pada setiap pemisahan node. Ini mengurangi korelasi antar pohon dan meningkatkan akurasi.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

importances = rf.feature_importances_
indices = np.argsort(importances)[-10:]

plt.figure(figsize=(10, 6))
plt.barh(range(len(indices)), importances[indices])
plt.yticks(range(len(indices)), [data.feature_names[i] for i in indices])
plt.title("Top 10 Feature Importances (Random Forest)")
plt.show()

## 4. Boosting

Boosting melatih model secara berurutan, di mana setiap model mencoba memperbaiki kesalahan model sebelumnya.

### Algoritma Populer:
- **AdaBoost**: Memberikan bobot lebih pada data yang salah diklasifikasikan.
- **Gradient Boosting**: Mengoptimalkan fungsi loss menggunakan gradient descent.
- **XGBoost / LightGBM**: Implementasi gradient boosting yang sangat cepat dan efisien.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3)
gb.fit(X, y)
print(f"Akurasi Gradient Boosting: {gb.score(X, y):.2f}")

## 5. Hyperparameter Tuning dan Cross-Validation

Bagaimana kita memilih nilai $K$ terbaik atau kedalaman pohon yang optimal?

### K-Fold Cross-Validation
Membagi data menjadi K bagian, melatih model pada K-1 bagian, dan mengujinya pada 1 bagian sisa secara bergantian.

### Grid Search vs Random Search
- **Grid Search**: Mencoba semua kombinasi parameter yang diberikan.
- **Random Search**: Mencoba kombinasi secara acak (lebih efisien untuk ruang parameter besar).

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance']
}

grid = GridSearchCV(KNeighborsClassifier(), param_grid, cv=5)
grid.fit(X_std, y)

print(f"Best Parameters: {grid.best_params_}")
print(f"Best Score: {grid.best_score_:.2f}")

## 6. Kesimpulan Bab 6

Statistical Machine Learning memindahkan fokus dari inferensi parameter ke akurasi prediksi.
- Random Forest adalah model "go-to" yang sangat kuat dan sulit dikalahkan tanpa tuning.
- Boosting seringkali memberikan akurasi tertinggi tetapi membutuhkan tuning yang lebih hati-hati.
- Selalu gunakan Cross-Validation untuk memastikan model Anda memiliki generalisasi yang baik.

### Konten Tambahan Detail: Bias-Variance Tradeoff

Ini adalah konsep paling fundamental dalam Machine Learning.
- **Bias**: Kesalahan karena asumsi model yang terlalu sederhana (Underfitting).
- **Variance**: Kesalahan karena model terlalu sensitif terhadap fluktuasi data pelatihan (Overfitting).
- **Goal**: Menemukan titik optimal di mana total error minimum.

#### Penjelasan Mendalam: Curse of Dimensionality

Dalam ruang berdimensi tinggi, data menjadi sangat jarang (*sparse*). 
Untuk KNN, ini berarti "tetangga terdekat" sebenarnya berada sangat jauh, sehingga konsep kemiripan menjadi kurang bermakna. 
Inilah mengapa reduksi dimensi (PCA) sering dilakukan sebelum menjalankan KNN.

#### Mekanisme Random Forest: Out-of-Bag (OOB) Error

Karena Random Forest menggunakan bootstrapping, sekitar 36.8% data tidak digunakan dalam setiap pohon. 
Data ini disebut "Out-of-Bag". Kita bisa menggunakan data ini untuk mengevaluasi model tanpa perlu dataset validasi terpisah.

#### Perbandingan Algoritma Boosting

| Algoritma | Karakteristik Utama | Keunggulan |
| :--- | :--- | :--- |
| AdaBoost | Fokus pada baris sulit | Sederhana, Interpretasi Bagus |
| Gradient Boosting | Optimasi Loss Function | Akurasi Tinggi |
| XGBoost | Regularisasi & Parallelism | Standar Kompetisi Kaggle |
| LightGBM | Leaf-wise growth | Sangat Cepat, Memory Efisien |
| CatBoost | Penanganan Kategorikal | Tanpa perlu One-Hot Encoding |

#### Pentingnya Feature Scaling

Algoritma berbasis jarak (KNN, SVM, K-Means) sangat memerlukan scaling. 
Namun, algoritma berbasis pohon (Decision Tree, Random Forest, XGBoost) bersifat *scale-invariant*, artinya hasilnya akan sama meskipun data tidak di-scale.

#### Teknik Tuning Lanjutan: Bayesian Optimization

Alih-alih mencoba parameter secara buta (Grid/Random Search), Bayesian Optimization membangun model probabilitas dari fungsi tujuan dan memilih parameter berikutnya secara cerdas untuk meminimalkan loss.

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier

# Model Extra Trees: Mirip Random Forest tapi lebih acak dalam memilih threshold pemisahan
et = ExtraTreesClassifier(n_estimators=100, random_state=42)
et.fit(X, y)
print(f"Akurasi Extra Trees: {et.score(X, y):.2f}")

#### Penutup Akhir Bab 6

Dengan menguasai Statistical Machine Learning, Anda telah memiliki senjata paling ampuh untuk menangani big data. 
Di bab terakhir (Bab 7), kita akan mempelajari teknik Unsupervised Learning untuk menemukan pola tanpa label.